# Iraqi Voice Studio — تدريب RVC v2 على Google Colab

دفتر Colab الرسمي لمنصة **استوديو الصوت العراقي**.

يُنفّذ عبر سكربت التثبيت الرسمي `scripts/colab/install_colab.sh`.

المتغيرات القابلة للتعديل في الخلية الأولى:
- `IRAQI_VOICE_REPO`: مستودع git للمشروع (افتراضيًا المستودع العام؛ اضبطه لنسختك الخاصة).
- `MOUNT_DRIVE`: ربط Google Drive (يُنفَّذ من نواة الجلسة هنا — لا يمكن ربطه من خلية bash).
- `DRIVE_DIR`: مجلد Drive الذي تُحفظ فيه النماذج ونقاط التحقق (افتراضي `RVC_Iraqi`).
- ضع ملفات البيانات الصوتية العراقية (10–60 دقيقة، نظيفة) في مجلد البيانات الذي تحدده خلية التجهيز (افتراضيًا `/content/drive/MyDrive/Voice`).
- **v1.3.1:** خلية التجهيز تتخطى الجاهز تلقائيًا، وخلية التدريب تستكمل دون `--dataset` ودون إعادة أي مرحلة — كل الإعدادات (epochs/batch/lr/جودة) تُطبَّق على الجلسة القائمة.

**ترتيب التشغيل:** 1 → 2 → 3 → 4، و5 اختيارية. شغّل الخلية الأخيرة (Snapshot) قبل انتهاء الجلسة أو عند أي انقطاع.

**ملاحظات مهمة:**
- نفق Cloudflare يُفعّل المصادقة تلقائيًا: كل نداءات API تعيد 401 حتى تدخل الرمز من تبويب **الإعدادات** في الصفحة (الرمز في `logs/auth.token`).
- عجلات CUDA لـ torch تُثبَّت تلقائيًا حسب درايفر GPU (cu128 للدرايفر 12.8+/13.x)، و`UV_NO_SYNC=1` يُضبط في الجلسة كي لا يعيدها `uv run` إلى CPU.

In [ ]:
# @title 1) الإعداد وربط Drive وفحص GPU (شغّلها أولًا)
IRAQI_VOICE_REPO = ""  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "RVC_Iraqi"  # @param {type:"string"}

import os
if IRAQI_VOICE_REPO:
    os.environ["IRAQI_VOICE_REPO"] = IRAQI_VOICE_REPO
os.environ["DRIVE_DIR"] = DRIVE_DIR

# ربط Drive يجب أن يحدث من نواة الجلسة (هذه الخلية) — من خلية bash يفشل دائمًا
# بخطأ AttributeError: 'NoneType' object has no attribute 'kernel'.
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("تعذّر ربط Drive:", e)
        print("أكمل بدون Drive — نقاط التحقق ستبقى في /content فقط (تُمسح بنهاية الجلسة).")

!nvidia-smi -L || echo "NO GPU"
!python3 --version
print("الخلية التالية: التثبيت الرسمي (سيكتشف Drive المرتبط تلقائيًا).")

In [ ]:
# @title 2) التثبيت الرسمي (uv + عجلات GPU + أصول + استعادة checkpoints إلى /content)
# ينسخ المستودع إلى /content/iraqi-voice-platform ثم يثبّت كل شيء:
# عجلة CUDA المناسبة للدرايفر، الأصول المسبقة، واستعادة runs/models من Drive إلى workspace المحلي.
import os
if os.environ.get("IRAQI_VOICE_REPO"):
    !rm -rf /content/iraqi-voice-platform
    !git clone --depth 1 "$IRAQI_VOICE_REPO" /content/iraqi-voice-platform
%cd /content/iraqi-voice-platform

!bash scripts/colab/install_colab.sh

# تفعيل توجيه الجلسة داخل النواة: مسارات Drive + UV_NO_SYNC الذي يمنع
# `uv run` من إرجاع عجلات torch إلى CPU. كل خلية !uv run لاحقة ترث هذا.
if os.path.exists("colab_wire_env.py"):
    exec(open("colab_wire_env.py").read())
    print(">>> توجيه الجلسة نشط داخل النواة <<<")
    print("RVC_RUNS_DIR =", os.environ.get("RVC_RUNS_DIR", "(محلي)"))
    print("UV_NO_SYNC   =", os.environ.get("UV_NO_SYNC", "0"))
else:
    print("تعذّر إنشاء colab_wire_env.py — تحقق من مخرجات الخلية أعلاه.")
# ---- حزمة الجودة (البحث المكثف — UPGRADE_REPORT_AR.md) ---------------------
# كل مكتبة اختيارية والكود يتحقق قبل استخدامها؛ أي فشل تثبيت لا يوقف شيءًا:
#   speechbrain  -> مقياس تشابه المتحدث ECAPA داخل مقياس الاختيار المركب
#   torchcrepe   -> طريقة F0 "crepe" (تقدير نغمة أقوى على بيانات صاخبة)
#   noisereduce  -> تنظيف الطيف الاختياري قبل التقطيع (--denoise)
!uv pip install --quiet speechbrain torchcrepe noisereduce || echo "(تعذّر تثبيت حزمة الجودة — التدريب يعمل بدونها بخط val_mel)"


In [ ]:
# @title 2b) (موصى به بشدة) قاعدة تدريب مسبق أقوى — TITAN أو Ov2Super
# البحث المكثف: القاعدة الرسمية مدرّبة على ~50 ساعة VCTK، بينما TITAN مدرّبة
# 1.01 مليون خطوة بجودة استوديو، وOv2Super مصممة أصلاً للبيانات الصغيرة
# (نتائج جيدة حتى من دقيقة واحدة!). لأي بيانات < ساعة: هذه أكبر رافعة جودة.
import os, json
PRETRAIN = "titan"  # @param ["titan", "ov2", "stock"]
SR = os.environ.get("RVC_SESSION_SR", "40k")  # يُثبَّت من خلية التجهيز 3 إن وُجدت
saved = None
for exp in ("iraqi_voice",):
    p = f"/content/iraqi-voice-platform/runs/{exp}/run_config.json"
    if os.path.isfile(p):
        try: saved = json.load(open(p)); break
        except Exception: pass
if saved:
    SR = saved.get("sample_rate", SR)
    print(f">>> SR مثبت على {SR} من الجلسة المجهزة {exp}")
!bash scripts/colab/get_pretrained.sh $PRETRAIN $SR


In [ ]:
# @title 3) تجهيز البيانات (تحقق → قص → F0 → ميزات → فهرس) — يتخطى الجاهز تلقائيًا
# التجهيز يتم مرة واحدة فقط؛ إعادة تشغيل هذه الخلية على جلسة مجهزة
# لن تعيد القص/استخراج F0/الميزات (SKIP_IF_PREPARED) إلا إذا غيّرت SR/F0
# أو فعّلت FORCE_REPREPARE أو ألغيت EXPERIMENT السابق.
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
import os, sys, json

EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_IN_DRIVE = "Voice"  # @param {type:"string"} (مجلد البيانات الخام — يُقرأ فقط عند التجهيز)
SR = "40k"  # @param ["32k", "40k", "48k"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm", "crepe"]
DENOISE = False  # @param {type:"boolean"} (تنظيف طيفي اختياري للبيانات الصاخبة — يتطلب حزمة الجودة)
LOUDNESS_NORM = True  # @param {type:"boolean"} (توحيد جهارة المقاطع — يُنصح به لأي مجموعة بيانات)
BASEPATH = "/content/"  # @param ["/content/drive/MyDrive/", "/content/"]
SKIP_IF_PREPARED = True  # @param {type:"boolean"} (تخطَّ التجهيز إذا الجلسة مجهزة بنفس SR/F0)
FORCE_REPREPARE = False  # @param {type:"boolean"} (فعّله فقط لإعادة التجهيز من الصفر — يمسح القديم!)

# متغيرات الخلية 2 (توجيه Drive وUV_NO_SYNC) مورثة تلقائيًا هنا لأنها في نفس النواة
PLATFORM = "/content/iraqi-voice-platform"
run_cfg_path = f"{PLATFORM}/runs/{EXPERIMENT}/run_config.json"
prepared = False
if os.path.isfile(run_cfg_path):
    saved = json.load(open(run_cfg_path))
    prepared = (saved.get("sample_rate") == SR
                and saved.get("f0_method") == F0_METHOD
                and os.path.isfile(f"{PLATFORM}/runs/{EXPERIMENT}/filelist.txt"))

if prepared and SKIP_IF_PREPARED and not FORCE_REPREPARE:
    n_lines = sum(1 for _ in open(f"{PLATFORM}/runs/{EXPERIMENT}/filelist.txt"))
    print(f">>> الجلسة '{EXPERIMENT}' مجهزة بالفعل (SR={SR}، F0={F0_METHOD}، "
          f"{n_lines} مقطعًا في filelist) — تم تخطي التجهيز.")
    print(">>> لتغيير SR أو F0_METHOD: فعّل FORCE_REPREPARE وأعد التشغيل (سيبدأ التدريب من الصفر).")
else:
    if FORCE_REPREPARE:
        print(">>> FORCE_REPREPARE — إعادة تجهيز كاملة (نقاط التحقق القديمة ستصبح غير صالحة).")
    !cd $PLATFORM && uv run rvc-platform data-check "$BASEPATH$DATASET_IN_DRIVE"
    denoise_flag = "--denoise" if DENOISE else "--no-denoise"
    loudness_flag = "--loudness" if LOUDNESS_NORM else "--no-loudness"
    !cd $PLATFORM && uv run rvc-platform prepare-dataset "$BASEPATH$DATASET_IN_DRIVE" "$EXPERIMENT" --sr "$SR" --f0-method "$F0_METHOD" --speech-filter 0.25 $denoise_flag $loudness_flag


In [ ]:
# @title 4) التدريب (يستكمل تلقائيًا من آخر Checkpoint عند إعادة التشغيل) — كل الخيارات
# v1.3.1: لا نمرر --dataset إطلاقًا على جلسة مجهزة — التدريب يستهلك التجهيز
# المحفوظ داخل runs/<EXPERIMENT>/ (مقاطع+F0+ميزات+filelist) ولا يعيد أي مرحلة،
# وكل الإعدادات أدناه تُطبَّق فعليًا على الجلسة القائمة (كانت تُتجاهل سابقًا).
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
import json, os, shlex

EXPERIMENT = "iraqi_voice"  # @param {type:"string"} (نفس اسم خلية التجهيز 3)

# ------------------------- الإعدادات الأساسية -------------------------------
EPOCHS = 300  # @param {type:"integer"} (الهدف الكلي؛ الاستكمال يُكمل إليه — 300 موصى بها لـ~40 دقيقة بيانات)
BATCH = 8  # @param {type:"integer"} (8 لبيانات ≥30 دقيقة؛ خفّضها إن نفدت الذاكرة)
LR = 0.0001  # @param {type:"number"} (معدل التعلم — لا ترفعه مع القواعد المسبقة)
SAVE_EVERY = 10  # @param {type:"integer"} (احفظ نقطة تحقق كل N حقبة)
GPU_INDEX = ""  # @param {type:"string"} (فارغ = اختيار تلقائي)
RESUME_FROM = ""  # @param {type:"string"} (اختياري: G_12600.pth أو latest — فارغ = الأحدث تلقائيًا)
CLEAN_START = False  # @param {type:"boolean"} (امسح كل نقاط التحقق وابدأ من الصفر!)
NO_PRETRAINED = False  # @param {type:"boolean"} (تخطي القاعدة المسبقة — ليس موصى به)

# ----------------- المرحلة 1: رافعات الجودة (UPGRADE_REPORT §1) -------------
EMA_DECAY = 0.999  # @param {type:"number"} (متوسط أوزان المولد؛ 0 = تعطيل)
STFT_WEIGHT = 2.0  # @param {type:"number"} (خسارة STFT متعددة الدقة؛ 0 = الهدف الأصلي)
USE_MSD = True  # @param {type:"boolean"} (تمييزات متعددة المقاييس — HiFi-GAN)
SPK_WEIGHT = 1.0  # @param {type:"number"} (وزن تشابه المتحدث ECAPA في اختيار الأفضل)
EARLY_STOP = 30  # @param {type:"integer"} (حقب صبر قبل الإيقاف التعاوني؛ 0 = تعطيل)
VAL_SPLIT = 0.10  # @param {type:"number"} (نسبة التحقق المحجوز للتقييم الموضوعي)
SAVE_BEST = True  # @param {type:"boolean"} (تتبّع وتصدير أفضل نموذج)

# ----------------- المرحلة 2: رافعات البحث الجديدة (§2.8+) ------------------
AUGMENT = True  # @param {type:"boolean"} (تعزيز لحظي: ضجيج/جهارة/حذف — للبيانات الصغيرة)
WARMUP = 1  # @param {type:"integer"} (حقب إحماء خطي لمعدل التعلم)
COSINE = 0.1  # @param {type:"number"} (ذيل جيب التمام؛ 0 = تعطيل، 0.1 موصى به)
D_LR_SCALE = 0.75  # @param {type:"number"} (معدل تعلم المميّز؛ 0.75 يحسّن الجودة المبكرة)

# ------------------------- أداء Colab ---------------------------------------
TORCH_COMPILE = "auto"  # @param ["auto", "on", "off"] (تسريع torch.compile على T4/L4)
NUM_WORKERS = 2  # @param {type:"integer"} (عدد عمّال تحميل البيانات؛ 0 = تلقائي)

# ------------------------- HuggingFace (اختياري) ----------------------------
HF_REPO = ""  # @param {type:"string"} (مستودع خاص owner/name لرفع النماذج)

# ---- فحص الجلسة + ملخص الاستكمال (SR وF0 مقفولان على الجلسة المجهزة) ----
PLATFORM = "/content/iraqi-voice-platform"
run_cfg_path = f"{PLATFORM}/runs/{EXPERIMENT}/run_config.json"
if not os.path.isfile(run_cfg_path):
    raise SystemExit(f"لا توجد جلسة '{EXPERIMENT}' — شغّل خلية التجهيز 3 أولًا.")
saved = json.load(open(run_cfg_path))
SR = saved.get("sample_rate", "40k")
F0_METHOD = saved.get("f0_method", "rmvpe")
ck_dir = f"{PLATFORM}/runs/{EXPERIMENT}/checkpoints"
state = {}
if os.path.isfile(f"{ck_dir}/state.json"):
    try: state = json.load(open(f"{ck_dir}/state.json"))
    except Exception: state = {}
g_pts = sorted([f for f in os.listdir(ck_dir) if f.startswith("G_") and f[2:-4].isdigit()]) if os.path.isdir(ck_dir) else []
print(f">>> الجلسة: {EXPERIMENT} | SR={SR} (مقفول) | F0={F0_METHOD} (مقفول)")
print(f">>> نقاط التحقق: {len(g_pts)} G_*.pth | آخر حقبة مسجلة: {state.get('epoch', 'بداية جديدة')}")
if state.get("epoch"):
    remaining = max(0, EPOCHS - int(state["epoch"]))
    print(f">>> الاستكمال: من الحقبة {state['epoch']} إلى {EPOCHS} (متبقٍ {remaining} حقبة) — لن يُعاد أي تجهيز.")
best_p = f"{ck_dir}/best.json"
if os.path.isfile(best_p):
    try:
        b = json.load(open(best_p))
        print(f">>> أفضل نموذج حتى الآن: epoch={b.get('epoch')} score={b.get('score')} ({b.get('metric')})")
    except Exception: pass

os.environ["RVC_TORCH_COMPILE"] = TORCH_COMPILE
os.environ["RVC_NUM_WORKERS"] = str(NUM_WORKERS)

# — فحص عجلة GPU قبل التدريب (شفاء تلقائي): إن رجعت torch إلى بناء CPU
#   (بسبب مزامنة عابرة) يُعاد تثبيت عجلة CUDA المناسبة للدرايفر تلقائيًا.
import subprocess as _sp
_probe = _sp.run(["/content/iraqi-voice-platform/.venv/bin/python",
                  "-c", "import torch; print(torch.__version__)"],
                 capture_output=True, text=True)
_tv = _probe.stdout.strip()
if _probe.returncode == 0 and "+cu" not in _tv and _tv:
    print(f">>> تحذير: torch {_tv} بناء CPU — إعادة تثبيت عجلة CUDA (env gpu cu128)...")
    import os as _o2
    _o2.chdir("/content/iraqi-voice-platform")
    _sp.run("uv run rvc-platform env gpu cu128", shell=True, check=False)
    _o2.chdir("/content")
else:
    print(f">>> عجلة GPU سليمة: torch {_tv}")
q = shlex.quote
flags = [
    f"--experiment {q(EXPERIMENT)}",
    f"--sr {SR}", f"--f0-method {q(F0_METHOD)}",
    f"--epochs {EPOCHS}", f"--batch {BATCH}", f"--lr {LR}",
    f"--save-every {SAVE_EVERY}",
    # المرحلة 1
    f"--ema-decay {EMA_DECAY}", f"--stft-weight {STFT_WEIGHT}",
    f"--spk-weight {SPK_WEIGHT}", f"--early-stop {EARLY_STOP}",
    f"--val-split {VAL_SPLIT}",
    "--save-best" if SAVE_BEST else "--no-save-best",
    "" if USE_MSD else "--no-msd",
    # المرحلة 2
    "--augment" if AUGMENT else "--no-augment",
    f"--warmup {WARMUP}", f"--cosine {COSINE}", f"--d-lr-scale {D_LR_SCALE}",
    # متنوعة
    f"--gpu {GPU_INDEX.strip()}" if GPU_INDEX.strip() else "",
    "--clean-start" if CLEAN_START else "",
    "--no-pretrained" if NO_PRETRAINED else "",
    f"--resume-from {q(RESUME_FROM)}" if RESUME_FROM.strip() else "",
    f"--hf-repo {q(HF_REPO)}" if HF_REPO.strip() else "",
]
cmd = "uv run rvc-platform train start " + " ".join(f for f in flags if f)
print(">>>", cmd)
!{cmd}  # النواة في جذر المنصة (من %cd في خلية 2) — وUV_NO_SYNC محفوظ في البيئة


In [ ]:
# @title 4b) (اختياري) تقييم موضوعي لأقوى نقطة تحقق (val_mel + تشابه المتحدث + composite)
# يقيّم أفضل نموذج مُصدَّر على البيانات المحجوزة ويكتب runs/<EXPERIMENT>/eval_report.json
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
!uv run rvc-platform train evaluate "$EXPERIMENT"

# مقارنة نقطة تدريب أخرى (مثلاً آخر checkpoint بدل الأفضل):
# !uv run rvc-platform train evaluate "$EXPERIMENT" --checkpoint G_48.pth


In [ ]:
# @title 4c) (جديد v1.3) حساء النقاط + التقييم الموضوعي — أفضل نموذج ممكن
# المتوسط المرجّح لآخر نقاط التحقق (Model-Soup/SWA — مرجح بالمقاييس اختياريًا)
# يقلل اهتزاز GAN ويحسّن الجودة مجانًا (بحث: Yazici et al., Wortsman et al.).
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
SOUP_K = 5  # @param {type:"integer"}
SOUP_WEIGHTED = True  # @param {type:"boolean"}
!uv run python scripts/train/make_soup.py "runs/$EXPERIMENT" --k $SOUP_K $( [ "$SOUP_WEIGHTED" = "True" ] && echo --weighted ) --export

# تقييم موضوعي: val_mel + تشابه المتحدث (ECAPA) + F0 — يقارن أفضل نقطة مقابل الحساء
!uv run rvc-platform train evaluate "$EXPERIMENT"

In [ ]:
# @title 5) (اختياري) واجهة الويب عبر نفق Cloudflare مؤقت
# النفق يُفعّل المصادقة تلقائيًا — الرمز في logs/auth.token:
#   !cat logs/auth.token
# انسخه وأدخله في تبويب الإعدادات داخل الصفحة، وإلا فكل نداءات API تعيد 401.
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
!uv run rvc-platform web --host 0.0.0.0 --tunnel

In [ ]:
# @title 6) Snapshot إلى Drive قبل انتهاء الجلسة (شغّلها يدويًا عند الانقطاع)
# يوقف التدريب تعاونيًا (مع حفظ checkpoint) ثم ينسخ runs/ و models/ إلى Drive.
# تتم المزامنة عبر ملفات مؤقتة وsnapshot_manifest.json للتحقق بعد الانقطاع.
# — حارس عجلات GPU: `uv run` العارية تعيد مزامنة uv.lock وتستبدل عجلات CUDA
#   بعجلات CPU (سلوك uv الموثّق) — UV_NO_SYNC=1 يمنع ذلك ويشغّل البيئة كما هي.
import os as _os; _os.environ["UV_NO_SYNC"] = "1"
!uv run rvc-platform train stop
!uv run rvc-platform colab snapshot
!uv run rvc-platform colab status
print(">>> حالّت الجلسة: البيانات محفوظة في Drive. عند جلسة جديدة: شغّل الخليتين 2 و4 فقط — سيُستكمل التدريب تلقائيًا. <<<")